In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter  
import os
from PyEMD import EMD
import warnings

plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
warnings.filterwarnings('ignore')


class EMDAnalyzer:
   
    def __init__(self, input_file: str, currency: str, n_imf_expected: int):
        self.input_file = input_file
        self.currency = currency  # Currency pair (e.g., EURUSD, GBPUSD)
        self.n_imf_expected = n_imf_expected  # Expected number of IMF components
    
    def emd_decompose_with_log(self, column='close', date_column='date', max_imf=None):
        # 1. Read data
        df = pd.read_csv(self.input_file, usecols=[date_column, column])
        
        # 2. Date parsing
        df[date_column] = pd.to_datetime(df[date_column])
        df = df.sort_values(by=date_column).reset_index(drop=True)
        
        # 3. Data type conversion
        df[column] = pd.to_numeric(df[column])
        
        # 4. Log transformation
        s = df[column].values
        log_s = np.log(s)
        
        # 5. EMD decomposition
        emd = EMD()
        emd.max_imf = max_imf if max_imf else self.n_imf_expected  
        emd_result = emd.emd(log_s)
        
        if isinstance(emd_result, tuple):
            imfs = emd_result[0]
            residual = emd_result[1]
        else:
            imfs = emd_result[:-1]
            residual = emd_result[-1]
        
        imfs = np.array(imfs)
        if imfs.ndim == 1:
            imfs = imfs.reshape(1, -1)
        n_imfs = imfs.shape[0]
        print(f"{self.currency} EMD decomposition completed: {n_imfs} actual IMF components (expected {self.n_imf_expected}) + 1 residual term")
        
        # 6. Construct result DataFrame
        result_df = pd.DataFrame({
            date_column: df[date_column],
            column: s,
            'log_' + column: log_s,
            'RES': residual
        })
        for i in range(n_imfs):
            result_df[f'IMF_{i+1}'] = imfs[i]
        
        return result_df, n_imfs


def plot_all_currencies():
    currency_config = [
        ('EURUSD', 'EURUSD_final.csv', 7),
        ('GBPUSD', 'GBPUSD_final.csv', 5),
        ('USDJPY', 'USDJPY_final.csv', 5),
        ('USDCNY', 'USDCNY_final.csv', 6)
    ]
    
    # 1. Create main figure
    fig = plt.figure(figsize=(16, 18), tight_layout=False)
    main_gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.1, wspace=0.15)
    emd_results = {}
    
    # 2. Iterate through each currency pair for processing
    for idx, (currency, file_path, n_imf_exp) in enumerate(currency_config):
        try:
            analyzer = EMDAnalyzer(file_path, currency, n_imf_exp)
            result_df, n_imfs = analyzer.emd_decompose_with_log(
                column='close', date_column='date', max_imf=n_imf_exp
            )
            emd_results[currency] = (result_df, n_imfs)
            
            output_csv = f'{currency}_IMFs_log_emd.csv'
            result_df.to_csv(output_csv, index=False)
            print(f"{currency} decomposition results saved to: {os.path.abspath(output_csv)}")
            
            total_subplots = 1 + n_imfs + 1
            sub_gs = gridspec.GridSpecFromSubplotSpec(
                total_subplots, 1, subplot_spec=main_gs[idx], 
                hspace=0.1, height_ratios=[1]*total_subplots  
            )
            
            date_col = 'date'
            log_col = 'log_close'
            imf_cols = [f'IMF_{i+1}' for i in range(n_imfs)]
            
            # 3. Plot original log signal (first subplot)
            ax0 = fig.add_subplot(sub_gs[0])
            ax0.plot(result_df[date_col], result_df[log_col], '#1f77b4', linewidth=1.0, label=f'log_{currency}')
            ax0.set_ylabel(f'log_close', fontsize=8)
            ax0.grid(True, alpha=0.3)
            ax0.set_title(f'{currency} EMD Decomposition (IMF={n_imfs})', fontsize=8, pad=8)
            ax0.tick_params(axis='y',  labelsize=8)
            ax0.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
            
            # 4. Plot all IMF components
            for i, imf_col in enumerate(imf_cols, 1):
                ax = fig.add_subplot(sub_gs[i], sharex=ax0)
                ax.plot(result_df[date_col], result_df[imf_col], 'red', linewidth=1.0, label=imf_col)
                ax.set_ylabel(imf_col, fontsize=8)
                ax.grid(True, alpha=0.3)
                ax.tick_params(axis='y',  labelsize=8)
                ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
                if i < n_imfs:
                    plt.setp(ax.get_xticklabels(), visible=False)
            
            # 5. Plot residual
            ax_last = fig.add_subplot(sub_gs[-1], sharex=ax0)
            ax_last.plot(result_df[date_col], result_df['RES'], 'green', linewidth=1.0, label='RES')
            ax_last.set_xlabel('Date (days)', fontsize=8)
            ax_last.set_ylabel('RES', fontsize=8)
            ax_last.grid(True, alpha=0.3)
            ax_last.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

            ax_last.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
            ax_last.tick_params(axis='x',  labelsize=8)
            ax_last.tick_params(axis='y',  labelsize=8)
            
        except Exception as e:
            print(f"{currency} processing failed: {str(e)}")
            ax_error = fig.add_subplot(main_gs[idx])
            ax_error.text(0.5, 0.5, f'{currency}\nProcess Failed\n{str(e)[:50]}...', 
                         ha='center', va='center', transform=ax_error.transAxes, fontsize=10)
            ax_error.set_title(f'{currency} EMD Decomposition', fontsize=10)
    
    # 6. Save and display the figure
    save_path = 'EMD_Complete.png'
    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )
    print(f"\nEMD plot saved to: {os.path.abspath(save_path)}")

    plt.show()
    plt.close()
    
    print("\n" + "="*60)
    print("Generated File List:")
    for currency, _, _ in currency_config:
        csv_file = f'{currency}_IMFs_log_emd.csv'
        if os.path.exists(csv_file):
            print(f"- {csv_file}")
    print(f"- {save_path}")
    print("="*60)


if __name__ == "__main__":
    plot_all_currencies()